# Test Hypothesis on First 50 Videos

Validate pipeline on Batch 1 (49 videos).
If IoU-F1 >= 0.30 → scale to 255.
Time: ~15 min

In [ ]:
# Cell 1: Setup
import os, json, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/standup4ai'
FEATURE_DIR = f'{BASE}/features_255'
WORK = '/content/test_hypothesis'
os.makedirs(WORK, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

feat_files = sorted([f for f in os.listdir(FEATURE_DIR) if '_features.npy' in f])
print(f'Available features: {len(feat_files)} videos')

In [ ]:
# Cell 2: Load Features
X_list, y_list, vids_list = [], [], []
for feat_file in feat_files:
    vid = feat_file.replace('_features.npy', '')
    lp = f'{FEATURE_DIR}/{vid}_labels.npy'
    if not os.path.exists(lp): continue
    X = np.load(f'{FEATURE_DIR}/{feat_file}')
    y = np.load(lp)
    X_list.append(X)
    y_list.append(y)
    vids_list.extend([vid] * len(y))

X_all = np.vstack(X_list)
y_all = np.concatenate(y_list)
groups = np.array(vids_list)
pos_rate = y_all.mean()

print(f'Total: {len(y_all)} words from {len(set(vids_list))} videos')
print(f'Positive rate: {100*pos_rate:.1f}% ({y_all.sum()} laugh)')
print(f'Feature dim: {X_all.shape[1]}')

In [ ]:
# Cell 3: Train FusionMLP (5-fold GroupKFold)
class FusionMLP(nn.Module):
    def __init__(self, input_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, 1))
    def forward(self, x): return self.net(x)

pos_weight_val = min((1.0 - pos_rate) / max(pos_rate, 1e-6), 3.0)
print(f'pos_weight: {pos_weight_val:.2f}')

gkf = GroupKFold(n_splits=5)
models, scalers, fold_results = [], [], []
all_probs = np.zeros(len(y_all))
test_indices = np.zeros(len(y_all), dtype=bool)

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_all, y_all, groups)):
    print(f'\n=== Fold {fold+1}/5 ===')
    Xtr, Xte = X_all[tr_idx], X_all[te_idx]
    ytr, yte = y_all[tr_idx], y_all[te_idx]

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr).astype(np.float32)
    Xte_s = scaler.transform(Xte).astype(np.float32)

    model = FusionMLP().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    pw = torch.tensor([pos_weight_val], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    Xtr_t = torch.tensor(Xtr_s).to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1).to(device)

    best_f1, patience, no_imp = 0, 5, 0
    for epoch in range(50):
        model.train()
        perm = torch.randperm(len(Xtr_t)).to(device)
        for i in range(0, len(Xtr_t), 256):
            idx = perm[i:i+256]
            opt.zero_grad()
            loss = criterion(model(Xtr_t[idx]), ytr_t[idx])
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            logits = model(torch.tensor(Xte_s).to(device)).squeeze().cpu().numpy()
            probs = 1/(1+np.exp(-logits))
            f = f1_score(yte, (probs >= 0.5).astype(int), zero_division=0)

        if f > best_f1:
            best_f1 = f; no_imp = 0
        else:
            no_imp += 1
        if no_imp >= patience: break

    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(Xte_s).to(device)).squeeze().cpu().numpy()
        probs_final = 1/(1+np.exp(-logits))
        all_probs[te_idx] = probs_final
        test_indices[te_idx] = True

    preds = (probs_final >= 0.5).astype(int)
    p = precision_score(yte, preds, zero_division=0)
    r = recall_score(yte, preds, zero_division=0)
    f = f1_score(yte, preds, zero_division=0)
    print(f'  F1={f:.4f} P={p:.4f} R={r:.4f} ({len(yte)} words)')

    models.append(model.cpu())
    scalers.append(scaler)
    fold_results.append({'fold': fold+1, 'f1': float(f), 'p': float(p), 'r': float(r)})

oof_preds = (all_probs[test_indices] >= 0.5).astype(int)
oof_f1 = f1_score(y_all[test_indices], oof_preds, zero_division=0)
mean_f1 = np.mean([r['f1'] for r in fold_results])
std_f1 = np.std([r['f1'] for r in fold_results])

print(f'\n{"="*50}')
print(f'OOF Word F1: {oof_f1:.4f}')
print(f'Mean Fold F1: {mean_f1:.4f} +/- {std_f1:.4f}')

In [ ]:
# Cell 4: IoU Segment Evaluation
LABEL_DIRS = [
    f'{BASE}/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train',
    f'{BASE}/seq-Standup4AI/dataset/en_uk/emnlp+jahak/val',
]

def find_label_path(vid):
    for ld in LABEL_DIRS:
        p = f'{ld}/{vid}.csv'
        if os.path.exists(p): return p
    return None

def bio_to_spans(df):
    spans, i = [], 0
    while i < len(df):
        lbl = str(df.iloc[i].get('label','')).strip()
        ts = eval(str(df.iloc[i]['timestamp']))
        if lbl == 'L':
            spans.append((float(ts[0]), float(ts[1])))
        elif lbl == 'B':
            st, en = float(ts[0]), float(ts[1])
            j = i + 1
            while j < len(df):
                nl = str(df.iloc[j].get('label','')).strip()
                if nl in ('I','L'):
                    en = float(eval(str(df.iloc[j]['timestamp']))[1]); j += 1
                else: break
            spans.append((st, en)); i = j - 1
        i += 1
    return spans

def span_iou(s1, s2):
    inter = max(0.0, min(s1[1],s2[1])-max(s1[0],s2[0]))
    union = max(s1[1],s2[1])-min(s1[0],s2[0])
    return inter/union if union > 0 else 0.0

def seg_f1(pred, gt, th=0.3):
    if not pred or not gt: return 0.0, 0.0, 0.0
    mp, mg = set(), set()
    for pi, ps in enumerate(pred):
        bi, bg = 0.0, -1
        for gi, gs in enumerate(gt):
            if gi in mg: continue
            iv = span_iou(ps, gs)
            if iv >= th and iv > bi: bi, bg = iv, gi
        if bg >= 0: mp.add(pi); mg.add(bg)
    tp = len(mp)
    p = tp/len(pred) if pred else 0.0
    r = tp/len(gt) if gt else 0.0
    return p, r, 2*p*r/(p+r) if (p+r)>0 else 0.0

def merge_segs(probs, ts, thr=0.5):
    spans, in_seg, start = [], False, 0.0
    for pr, (t0,t1) in zip(probs, ts):
        if pr >= thr and not in_seg: in_seg, start = True, t0
        elif pr < thr and in_seg: in_seg = False; spans.append((start, t0))
    if in_seg: spans.append((start, ts[-1][1]))
    return spans

THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]
RESULTS = {th: [] for th in THRESHOLDS}
per_video_iou = []

unique_vids = sorted(set(vids_list))
for vid in unique_vids:
    mask = np.array([v == vid for v in vids_list]) & test_indices
    if mask.sum() == 0: continue

    vid_probs = all_probs[mask]
    lp = find_label_path(vid)
    if not lp: continue
    df_full = pd.read_csv(lp)

    timestamps = []
    for _, row in df_full.iterrows():
        try:
            t = eval(str(row['timestamp']))
            timestamps.append((float(t[0]), float(t[1])))
        except: pass

    n_valid = min(len(vid_probs), len(timestamps))
    if n_valid < 2: continue

    gt_spans = bio_to_spans(df_full)
    if not gt_spans: continue

    pred_spans = merge_segs(vid_probs[:n_valid], timestamps[:n_valid], 0.5)

    row = {'vid': vid, 'n_pred': len(pred_spans), 'n_gt': len(gt_spans)}
    for th in THRESHOLDS:
        p, r, f = seg_f1(pred_spans, gt_spans, th)
        row[f'f_{th}'] = round(f, 4)
        RESULTS[th].append({'vid': vid, 'p': p, 'r': r, 'f': f})
    per_video_iou.append(row)

print('='*65)
print('IoU SEGMENT-LEVEL EVALUATION — OOF Predictions')
print(f'N = {len(per_video_iou)} videos | Baseline: F1=0.51 @ IoU=0.2')
print('='*65)
summary_iou = {}
for th in THRESHOLDS:
    rs = RESULTS[th]
    if not rs: continue
    fm = np.mean([x['f'] for x in rs])
    pm = np.mean([x['p'] for x in rs])
    rm = np.mean([x['r'] for x in rs])
    summary_iou[th] = {'f1': float(fm), 'p': float(pm), 'r': float(rm)}
    print(f'  IoU>={th:.1f}: F1={fm:.4f} P={pm:.4f} R={rm:.4f}')

print('\nTop per-video:')
for row in sorted(per_video_iou, key=lambda x: x.get('f_0.3',0), reverse=True)[:8]:
    print(f"  {row['vid']:<18} gt={row['n_gt']:>3} pd={row['n_pred']:>4} F1@0.3={row.get('f_0.3',0):.4f}")

In [ ]:
# Cell 5: Verdict + Save
iou_02 = summary_iou.get('0.2', {}).get(0.2, {}).get('f1', 0)
iou_02 = summary_iou.get(0.2, summary_iou.get('0.2', {})).get('f1', 0)

print('\n' + '='*60)
print('HYPOTHESIS TEST RESULTS')
print('='*60)
print(f'Word-level OOF F1: {oof_f1:.4f}')
print(f'IoU-F1 @ 0.2:      {iou_02:.4f}')
print(f'IoU-F1 @ 0.3:      {summary_iou.get(0.3, {}).get("f1", 0):.4f}')
print(f'Baseline:          F1=0.51 @ IoU=0.2')
print()

if iou_02 >= 0.51:
    verdict = 'beats_baseline'
    print('🎉 BEATS BASELINE! Scale to 255.')
elif iou_02 >= 0.30:
    verdict = 'promising'
    print('✅ Promising. Scale to 255.')
elif iou_02 >= 0.15:
    verdict = 'weak'
    print('⚠️ Weak. Debug before scaling.')
else:
    verdict = 'poor'
    print('❌ Poor. Investigate.')

import shutil
results_out = {
    'n_videos': len(unique_vids),
    'n_words': int(len(y_all)),
    'positive_rate': float(pos_rate),
    'oof_word_f1': float(oof_f1),
    'fold_results': fold_results,
    'iou_summary': {str(k): v for k,v in summary_iou.items()},
    'verdict': verdict,
}
with open('/content/test_results.json', 'w') as f:
    json.dump(results_out, f, indent=2)
os.makedirs(f'{BASE}/models', exist_ok=True)
shutil.copy('/content/test_results.json', f'{BASE}/models/hypothesis_test_batch1.json')
print(f'Saved: {BASE}/models/hypothesis_test_batch1.json')